# Framework RAG para interpretación de leyes chilenas — Demo end-to-end

Este notebook ejecuta, de punta a punta, el framework descrito en la arquitectura acordada:

1. **Ingesta y preprocesamiento**: PDF → Markdown, chunking jerárquico (Título/Artículo/Inciso), indexación en ChromaDB.
2. **Indexación híbrida**: denso (BGE-M3) + disperso (BM25).
3. **Recuperación (query time)**: pregunta → ruteo semántico (Qwen3) → búsqueda densa + dispersa → fusión RRF → reranker (BGE FlagReranker) → contexto top-k con cita `Ley/Artículo`.
4. **Generación**: respuesta en lenguaje natural citando siempre la fuente legal (Qwen3).

### ⚠️ Nota importante sobre el modo de ejecución

Este notebook corre en un entorno sandbox **sin acceso a Internet para descargar modelos** (HuggingFace y toda API externa están bloqueadas por política de egress). Por eso el framework tiene **dos modos intercambiables**, controlados por una sola variable (`RAG_MODE`):

| Componente | `production` (arquitectura pedida) | `lite` (lo que corre en ESTE notebook) |
|---|---|---|
| Embeddings densos | BAAI/bge-m3 (HuggingFace) | TF-IDF + SVD (scikit-learn, con stemming en español) |
| Reranker | BAAI/bge-reranker-v2-m3 (FlagEmbedding) | Heurística léxica (Jaccard + coincidencia de N° de artículo) |
| Router semántico | Qwen3 (HF / Ollama) | TF-IDF contra el catálogo temático de leyes |
| Generador de respuesta | Qwen3 (HF / Ollama) | Extractivo (compone la respuesta citando los chunks tal cual, sin LLM) |
| BM25 | rank_bm25 | rank_bm25 (idéntico en ambos modos) |
| ChromaDB | idéntico en ambos modos | idéntico en ambos modos |

Todo el código de producción (clases `BGEM3Embedder`, `BGERerankerProduction`, `QwenRouterProduction`, `QwenGeneratorProduction`) **ya está escrito** en `leyes_rag/` y listo para usarse — basta correr `RAG_MODE=production` en una máquina con GPU/Internet. Ver la sección final "Cómo pasar a producción".

El objetivo de este notebook es que puedas **probar el pipeline completo hoy, actuando como distintos clientes PYME**, y decirme qué está mal para corregirlo juntos antes de invertir en la infraestructura de producción.


In [1]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import os
os.environ.setdefault("RAG_MODE", "lite")

from leyes_rag.config import SETTINGS, LAW_CATALOG
print("Modo RAG activo:", SETTINGS.mode)
print()
print("Leyes en el catálogo:")
for law in LAW_CATALOG:
    print(f"  - {law['nombre_corto']}  ({law['file']})")


Modo RAG activo: lite

Leyes en el catálogo:
  - Ley 21.719 - Protección de Datos Personales  (Ley-21719_Proteccion_Datos.pdf)
  - Ley 19.628 - Protección de la Vida Privada  (LEY-19628_Proteccion_Vida_Privada.pdf)
  - Ley 20.416 - Estatuto PYME  (Ley-20416_Especial_Pymes.pdf)
  - Ley 20.285 - Acceso a la Información Pública  (LEY-20285_Acceso_Informacion_Publica.pdf)
  - Ley 19.496 - Protección de los Derechos de los Consumidores  (Ley-19496_Proteccion_Derechos_Consumidores.pdf)
  - Constitución Política de la República  (DTO-100_Constitucion_Politica_Republica.pdf)


> **¿Ya corriste este notebook antes y solo quieres hacer consultas nuevas?**
> Las celdas de las Fases 1 y 2 (abajo) leen los PDF, los convierten a Markdown y RECONSTRUYEN los índices desde cero — solo hace falta correrlas una vez, o cuando cambien las leyes fuente. Para consultar sin reprocesar nada, salta directo a la sección **"Servicio RAG completo"** y reemplaza esa celda por:
> ```python
> from leyes_rag.rag_service import LeyesRAGService
> svc = LeyesRAGService()   # carga data/chroma_db, bm25.pkl y embedder_state.pkl ya existentes
> ```
> o, fuera del notebook, usa `python scripts/02_query_cli.py "tu pregunta"` — es justamente para eso.

## Fase 1 — Ingesta y preprocesamiento

Convertimos cada PDF a texto limpio (removiendo encabezados repetidos de página y anotaciones marginales de "historia de la ley" propias del formato de la Biblioteca del Congreso Nacional), reconstruimos los párrafos por sangría, y parseamos la jerarquía **Capítulo → Título → Párrafo → Artículo → Inciso**.

La salida son dos cosas:
- Un archivo **Markdown legible por humano** por cada ley (`data/markdown/*.md`), para poder revisar visualmente que el parseo fue correcto.
- Una lista de **chunks jerárquicos** con su metadata completa (ley, título, artículo, inciso, página, cita) lista para indexar.


In [2]:
from leyes_rag.ingest.pdf_extract import extract_raw_paragraphs
from leyes_rag.ingest.chunking import parse_paragraphs_to_articles, build_chunks, render_markdown
from leyes_rag.config import RAW_LEYES_DIR, MARKDOWN_DIR

MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

all_chunks = []
stats = []
for law in LAW_CATALOG:
    paragraphs = extract_raw_paragraphs(RAW_LEYES_DIR / law["file"])
    preamble, articles = parse_paragraphs_to_articles(paragraphs)
    chunks = build_chunks(law["id"], articles, SETTINGS.chunking)
    all_chunks.extend(chunks)

    md_text = render_markdown(law["id"], preamble, articles)
    (MARKDOWN_DIR / f"{law['id']}.md").write_text(md_text, encoding="utf-8")

    stats.append((law["nombre_corto"], len(articles), len(chunks)))

import pandas as pd
df_stats = pd.DataFrame(stats, columns=["Ley", "Artículos detectados", "Chunks generados"])
df_stats


,Ley,Artículos detectados,Chunks generados
0,Ley 21.719 - Protección de Datos Personales,80,355
1,Ley 19.628 - Protección de la Vida Privada,27,62
2,Ley 20.416 - Estatuto PYME,57,142
3,Ley 20.285 - Acceso a la Información Pública,67,155
4,Ley 19.496 - Protección de los Derechos de los...,147,388
5,Constitución Política de la República,169,411


### Vista previa del Markdown generado (Ley 20.416 — Estatuto PYME)

In [3]:
from IPython.display import Markdown
preview = (MARKDOWN_DIR / "pymes_2010.md").read_text(encoding="utf-8")
Markdown(preview[:2500] + "\n\n*(...)*")


# Ley 20.416 - Estatuto PYME

> FIJA NORMAS ESPECIALES PARA LAS EMPRESAS DE MENOR TAMAÑO MINISTERIO DE ECONOMÍA, FOMENTO Y RECONSTRUCCIÓN; SUBSECRETARÍA DE ECONOMÍA, FOMENTO Y RECONSTRUCCIÓN LEY NÚM. 20.416 FIJA NORMAS ESPECIALES PARA LAS EMPRESAS DE MENOR TAMAÑO Teniendo presente que el H. Congreso Nacional ha dado su aprobación al siguiente Proyecto de ley:


**Artículo Primero — Objetivo**

La presente ley tiene por objeto facilitar el desenvolvimiento de las empresas de menor tamaño, mediante la adecuación y creación de normas regulatorias que rijan su iniciación, funcionamiento y término, en atención a su tamaño y grado de desarrollo.


**Artículo Segundo — Sujeto**

Para los efectos de esta ley, se entenderá por empresas de menor tamaño las microempresas, pequeñas empresas y medianas empresas.
Son microempresas aquellas empresas cuyos ingresos anuales por ventas y servicios y otras actividades del giro no hayan superado las 2.400 unidades de fomento en el último año calendario; pequeñas empresas, aquellas cuyos ingresos anuales por ventas, servicios y otras actividades del giro sean superiores a 2.400 unidades de fomento y no exceden de 25.000 unidades de fomento en el último año calendario, y medianas empresas, aquellas cuyos ingresos anuales por ventas, servicios y otras actividades del giro sean superiores a 25.000 unidades de fomento y no exceden las 100.000 unidades de fomento en el último año calendario.
El valor de los ingresos anuales por ventas y servicios y otras actividades del giro señalado en el inciso anterior se refiere al monto total de éstos, para el año calendario anterior, descontado el valor correspondiente al impuesto al valor agregado y a los impuestos específicos que pudieren aplicarse.
Si la empresa hubiere iniciado actividades el año calendario anterior, los límites a que se refieren los incisos precedentes se establecerán considerando la proporción de ingresos que representen los meses en que el contribuyente haya desarrollado actividades.
Para los efectos de la determinación de los ingresos, las fracciones de meses se considerarán como meses completos.
Dentro del rango máximo de 100.000 unidades de fomento establecido en el inciso segundo, el Presidente de la República, mediante decreto supremo del Ministro de Economía, Fomento y Reconstrucción y previa consulta o a requerimiento del Consejo Consultivo de la Empresa de Menor Tamaño, podrá modificar la clasificación de las Empresas de Menor Tamaño o establecer factores o ind

*(...)*

### Ejemplo de chunk jerárquico (lo que efectivamente se guarda en ChromaDB, con su metadata)

In [4]:
import json
ejemplo = next(c for c in all_chunks if c.law_id == "consumidores_1997" and c.articulo == "3º")
print("chunk_id  :", ejemplo.chunk_id)
print("citation  :", ejemplo.citation)
print("metadata  :", json.dumps({k:v for k,v in ejemplo.__dict__.items() if k not in ("text",)}, ensure_ascii=False, indent=2))
print()
print("texto     :")
print(ejemplo.text)


chunk_id  : consumidores_1997__art3__1
citation  : Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º, Inciso 1°
metadata  : {
  "chunk_id": "consumidores_1997__art3__1",
  "law_id": "consumidores_1997",
  "law_name": "Ley 19.496 - Protección de los Derechos de los Consumidores",
  "law_number": "19.496",
  "source_file": "Ley-19496_Proteccion_Derechos_Consumidores.pdf",
  "capitulo": null,
  "titulo": "II",
  "parrafo": "1º",
  "articulo": "3º",
  "inciso_range": "inciso 1°",
  "citation": "Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º, Inciso 1°",
  "page_no": 4
}

texto     :
[Ley 19.496 - Protección de los Derechos de los Consumidores] Título II (Disposiciones generales) > Párrafo 1º (Los derechos y deberes del consumidor) — Artículo 3º (inciso 1°)
Son derechos y deberes básicos del consumidor:


## Fase 2 — Indexación híbrida

- **Denso**: se vectoriza cada chunk con el `Embedder` activo (TF-IDF+SVD en modo lite, BGE-M3 en producción) y se guarda en **ChromaDB** con similitud coseno.
- **Disperso**: se indexa el mismo corpus con **BM25** (rank_bm25), usando un tokenizador con stemming en español (para que "eliminar" y "eliminación" compartan raíz léxica).

Ambos índices se construyen sobre los mismos `chunk_id`, lo que permite fusionarlos después con RRF.


In [5]:
from leyes_rag.indexing.vector_store import build_vector_store
from leyes_rag.indexing.sparse_index import build_bm25_index

embedder = build_vector_store(all_chunks)
bm25 = build_bm25_index(all_chunks)  # BM25Index, ya listo para el pipeline (.search())

print(f"Embedder activo: {embedder.name} (dim={embedder.dim})")
print(f"Chunks indexados: {len(all_chunks)}")


Embedder activo: tfidf-lite (dim=384)
Chunks indexados: 1513


## Fase 3 — Recuperación (query time), paso a paso

Antes de correr el servicio completo, veamos **una consulta en modo "caja de vidrio"**: qué hace el router, qué trae cada búsqueda por separado, cómo las fusiona RRF, y cómo las reordena el reranker.


In [6]:
from leyes_rag.retrieval.pipeline import RetrievalPipeline
from leyes_rag.notebook_utils import show_trace, show_answer

pipeline = RetrievalPipeline(embedder, bm25)

trace = pipeline.retrieve("¿Cuáles son los derechos básicos del consumidor?", top_k_final=5)
show_trace(trace)


### 🔎 Trace de recuperación para: *"¿Cuáles son los derechos básicos del consumidor?"*

**1. Ruteo semántico** → leyes candidatas: `['consumidores_1997', 'constitucion_2005']`
> Router léxico -> Ley 19.496 - Protección de los Derechos de los Consumidores (0.30), Constitución Política de la República (0.09)

**2. Búsqueda densa (coseno)** — top 5 de 15:
- `0.678` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º, Inciso 1°
- `0.544` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 4º
- `0.539` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º, Literal f)
- `0.465` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 2º, Inciso 1°
- `0.394` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 2º, Literal b)

**3. Búsqueda dispersa (BM25)** — top 5 de 15:
- `12.489` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º, Inciso 1°
- `12.160` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 17 B, Literal d)
- `9.119` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 1º, Numeral 4
- `8.380` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 1º, Numeral 3
- `7.942` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 1º, Inciso 9°

**4. Fusión RRF** — top 5 de 12:
- `0.0328` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º, Inciso 1°
- `0.0308` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 2º, Inciso 1°
- `0.0303` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 2º, Literal b)
- `0.0303` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 1º, Inciso 9°
- `0.0298` Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 4º

**5. Reranker** — contexto final (5 chunks):
- `1.035` **Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º, Inciso 1°** (dense#1, sparse#1, rrf=0.0328)
- `0.525` **Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 2º, Inciso 1°** (dense#4, sparse#6, rrf=0.0308)
- `0.357` **Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 2º, Literal b)** (dense#5, sparse#7, rrf=0.0303)
- `0.261` **Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 1º, Inciso 9°** (dense#7, sparse#5, rrf=0.0303)
- `0.223` **Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 4º** (dense#2, sparse#13, rrf=0.0298)

## Servicio RAG completo (retrieval + generación con cita)

`LeyesRAGService` une todo el pipeline: ruteo → híbrido → RRF → rerank → generación de respuesta citando `Ley/Artículo`. En modo lite, la "generación" es **extractiva** (muestra el texto real de los chunks, sin redactar con un LLM) para que la respuesta sea 100% trazable a la ley mientras no tenemos Qwen3 corriendo.


In [7]:
from leyes_rag.rag_service import LeyesRAGService

svc = LeyesRAGService()


## 🧑‍💼 Simulación de consultas como clientes PYME

Probamos el sistema actuando como distintas PYMES chilenas con preguntas reales sobre cada una de las 6 leyes del corpus.


In [8]:
result = svc.ask('Un cliente compró una impresora en mi tienda y a los 5 días la quiere devolver porque "no le gustó", sin que esté fallada. ¿Estoy obligado a devolverle el dinero?', top_k_final=4)
show_answer(result)

### 💬 Pregunta: *"Un cliente compró una impresora en mi tienda y a los 5 días la quiere devolver porque "no le gustó", sin que esté fallada. ¿Estoy obligado a devolverle el dinero?"*

**Leyes consultadas por el router:** consumidores_1997

**Respuesta:**

Esto es lo que encontré en las leyes indexadas relacionado con tu pregunta (modo LITE: extracto textual sin redacción por LLM; verifica siempre citando la fuente):



• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º bis, Inciso 6°–inciso 7°) Si el consumidor ejerciera el derecho consagrado en este artículo, el proveedor estará obligado a devolverle las sumas abonadas, sin retención de gastos, a la mayor brevedad posible y, en cualquier caso, antes de cuarenta y cinco días siguientes a la comunicación del retracto. Tratándose de servicios, la devolución sólo comprenderá aquellas sumas abonadas que no correspondan a servicios ya prestados al consumidor a la fecha del retracto. Deberán restituirse en buen estado los elementos originales del embalaje, como las etiquetas, certificados de garantía, manuales de uso, cajas, elementos de protección o su valor respectivo, previamente informado.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º ter, Inciso 3°–inciso 4°) En ningún caso la institución educacional podrá retener con posterioridad a este retracto los dineros pagados ni los documentos de pago o crédito otorgados en respaldo del período educacional respectivo, debiendo devolverlos todos en el plazo de 10 días desde que se ejerza el derecho a retracto. En el evento de haberse otorgado mandato general para hacer futuros cobros, éste quedará revocado por el solo ministerio de la ley desde la fecha de la renuncia efectiva del alumno al servicio educacional. El prestador del servicio se abstendrá de negociar o endosar los documentos recibidos, antes del plazo señalado en el inciso primero. No obstante lo dispuesto en el inciso anterior, la institución de educación superior estará facultada para retener, por concepto de costos de administración, un monto de la matrícula, que no podrá exceder al uno por ciento del arancel anual del programa o carrera.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 14) Cuando con conocimiento del proveedor se expendan productos con alguna deficiencia, usados o refaccionados o cuando se ofrezcan productos en cuya fabricación o elaboración se hayan utilizado partes o piezas usadas, se deberán informar de manera expresa las circunstancias antes mencionadas al consumidor, antes de que éste decida la operación de compra. Será bastante constancia el usar en los propios artículos, en sus envoltorios, en avisos o carteles visibles en sus locales de atención al público las expresiones "segunda selección", "hecho con materiales usados" u otras equivalentes. El cumplimiento de lo dispuesto en el inciso anterior eximirá al proveedor de las obligaciones derivadas del derecho de opción que se establece en los artículos 19 y 20, sin perjuicio de aquellas que hubiera contraído el proveedor en virtud de la garantía otorgada al producto.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 56, Inciso 1°–inciso 3°) El servicio de atención al cliente requerido para dar cumplimiento a la condición dispuesta en el número 2 del inciso primero del artículo 55 será organizado por los proveedores indicados en este Título, en forma exclusiva o conjunta, y será gratuito para el consumidor que haya suscrito un contrato de adhesión de los señalados en el inciso segundo del artículo 55, con un proveedor que cuente con el sello SERNAC. El servicio de atención al cliente deberá responder fundadamente los reclamos de los consumidores, en el plazo de diez días hábiles contado desde su presentación. Esta respuesta se comunicará al consumidor por escrito o mediante cualquier medio físico o tecnológico y se enviará copia de ella al Servicio Nacional del Consumidor. El proveedor deberá dar cumplimiento a lo señalado en la respuesta del servicio de atención al cliente en el plazo de cinco días hábiles, contado desde la comunicación al consumidor.

**Fuentes citadas:**
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º bis, Inciso 6°–inciso 7°
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 3º ter, Inciso 3°–inciso 4°
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 14
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 56, Inciso 1°–inciso 3°

In [9]:
result = svc.ask('Tengo una pequeña empresa de software con 8 trabajadores. ¿Qué se considera legalmente una "empresa de menor tamaño" y qué beneficios especiales me aplican por ser PYME?', top_k_final=4)
show_answer(result)

### 💬 Pregunta: *"Tengo una pequeña empresa de software con 8 trabajadores. ¿Qué se considera legalmente una "empresa de menor tamaño" y qué beneficios especiales me aplican por ser PYME?"*

**Leyes consultadas por el router:** pymes_2010

**Respuesta:**

Esto es lo que encontré en las leyes indexadas relacionado con tu pregunta (modo LITE: extracto textual sin redacción por LLM; verifica siempre citando la fuente):



• (Ley 20.416 - Estatuto PYME, Artículo Tercero, Inciso 1°–inciso 4°) El Ministerio de Economía, Fomento y Reconstrucción deberá impulsar el desarrollo de las empresas de menor tamaño y facilitarles la utilización de los instrumentos de fomento dispuestos por los órganos del Estado. Le corresponderá a la Subsecretaría de Economía, Fomento y Reconstrucción generar coordinaciones para que, en conjunto con los ministerios sectoriales, se formulen las políticas y planes de fomento considerando las particularidades de las empresas de menor tamaño. Asimismo, le corresponderá impulsar con sus servicios dependientes o relacionados una política general para la mejor orientación, coordinación y fomento del desarrollo de las empresas de menor tamaño, así como realizar un seguimiento de las respectivas políticas y programas y generar las condiciones para el acceso de estas empresas a fuentes útiles de información, contribuyendo a la mejor utilización de los instrumentos de fomento disponibles para ellas. Créase la División de empresas de menor tamaño en la Subsecretaría de Economía, Fomento y Reconstrucción.

• (Ley 20.416 - Estatuto PYME, Artículo Cuarto, Inciso 1°–inciso 2°) Del Consejo Nacional Consultivo de la Empresa de Menor Tamaño. Créase el Consejo Nacional Consultivo de la Empresa de Menor Tamaño, en adelante y para todos los efectos de esta ley, "el Consejo", cuya función será asesorar al Ministerio de Economía, Fomento y Reconstrucción en la proposición de políticas y coordinación de esfuerzos de los sectores público y privado, destinados a promover una adecuada participación de las empresas de menor tamaño en la economía nacional. El Consejo estará integrado por los siguientes miembros:

• (Ley 20.416 - Estatuto PYME, Artículo Primero (transitorio)) Dentro de los dos primeros años de vigencia de la presente ley, la Subsecretaría de Economía y Empresas de Menor Tamaño, hará una revisión de la reglamentación actualmente aplicable a las Empresas de Menor Tamaño, con la finalidad de proponer a los órganos públicos competentes las adecuaciones que sean necesarias para el cumplimiento del objetivo señalado en el ARTÍCULO PRIMERO de la presente ley. Dicha revisión dará origen a un compendio de normas aplicables a las empresas de menor tamaño, el cual será puesto a disposición de los usuarios en la página web del Ministerio. Asimismo, y dentro de los seis meses siguientes a la revisión reglamentaria señalada en el inciso anterior, se establecerá una tipología general que permita graduar las sanciones según la gravedad de los ilícitos susceptibles de fiscalización.

• (Ley 20.416 - Estatuto PYME, Artículo Segundo, Inciso 4°–inciso 6°) Si la empresa hubiere iniciado actividades el año calendario anterior, los límites a que se refieren los incisos precedentes se establecerán considerando la proporción de ingresos que representen los meses en que el contribuyente haya desarrollado actividades. Para los efectos de la determinación de los ingresos, las fracciones de meses se considerarán como meses completos. Dentro del rango máximo de 100.000 unidades de fomento establecido en el inciso segundo, el Presidente de la República, mediante decreto supremo del Ministro de Economía, Fomento y Reconstrucción y previa consulta o a requerimiento del Consejo Consultivo de la Empresa de Menor Tamaño, podrá modificar la clasificación de las Empresas de Menor Tamaño o establecer factores o indicadores adicionales para su categorización.

**Fuentes citadas:**
- Ley 20.416 - Estatuto PYME, Artículo Tercero, Inciso 1°–inciso 4°
- Ley 20.416 - Estatuto PYME, Artículo Cuarto, Inciso 1°–inciso 2°
- Ley 20.416 - Estatuto PYME, Artículo Primero (transitorio)
- Ley 20.416 - Estatuto PYME, Artículo Segundo, Inciso 4°–inciso 6°

In [10]:
result = svc.ask('Quiero enviar correos publicitarios a los clientes que compraron en mi tienda. ¿Necesito su autorización para usar sus datos personales con fines de marketing?', top_k_final=4)
show_answer(result)

### 💬 Pregunta: *"Quiero enviar correos publicitarios a los clientes que compraron en mi tienda. ¿Necesito su autorización para usar sus datos personales con fines de marketing?"*

**Leyes consultadas por el router:** proteccion_datos_2024, vida_privada_1999

**Respuesta:**

Esto es lo que encontré en las leyes indexadas relacionado con tu pregunta (modo LITE: extracto textual sin redacción por LLM; verifica siempre citando la fuente):



• (Ley 21.719 - Protección de Datos Personales, Artículo 3°, Literal b)) b) Principio de finalidad. Los datos personales deben ser recolectados con fines específicos, explícitos y lícitos. El tratamiento de los datos personales debe limitarse al cumplimiento de estos fines. En aplicación de este principio, no se pueden tratar los datos personales con fines distintos a los informados al momento de la recolección, salvo que el tratamiento sea para fines compatibles con los autorizados originalmente; que exista una relación contractual o precontractual entre el titular y el responsable que justifique el tratamiento de los datos con una finalidad distinta, siempre que se enmarque dentro de los fines del contrato o sea coherente con las tratativas o negociaciones previas a la celebración del mismo; que el titular otorgue nuevamente su consentimiento, y cuando lo disponga la ley.

• (Ley 21.719 - Protección de Datos Personales, Artículo 8°, Literal b)) b) Si el tratamiento se realiza exclusivamente con fines de mercadotecnia o marketing directo de bienes, productos o servicios, incluida la elaboración de perfiles, de conformidad con el artículo 8° bis.

• (Ley 21.719 - Protección de Datos Personales, Artículo 11, Literal b)) b) Indicación de un domicilio o una dirección de correo electrónico o de otro medio equivalente para comunicar la respuesta.

• (Ley 21.719 - Protección de Datos Personales, Artículo 3°, Literal c)) c) Principio de proporcionalidad. Los datos personales que se traten deben limitarse estrictamente a aquéllos que resulten necesarios, adecuados y pertinentes en relación con los fines del tratamiento. Los datos personales pueden ser conservados sólo por el período de tiempo que sea necesario para cumplir con los fines del tratamiento, luego de lo cual deben ser suprimidos o anonimizados, sin perjuicio de las excepciones que establezca la ley. Un período de tiempo mayor requiere autorización legal o consentimiento del titular.

**Fuentes citadas:**
- Ley 21.719 - Protección de Datos Personales, Artículo 3°, Literal b)
- Ley 21.719 - Protección de Datos Personales, Artículo 8°, Literal b)
- Ley 21.719 - Protección de Datos Personales, Artículo 11, Literal b)
- Ley 21.719 - Protección de Datos Personales, Artículo 3°, Literal c)

In [11]:
result = svc.ask('Le vendo insumos de oficina a una municipalidad. ¿Puedo pedir información pública sobre cómo se adjudicó la licitación en la que participé y perdí?', top_k_final=4)
show_answer(result)

### 💬 Pregunta: *"Le vendo insumos de oficina a una municipalidad. ¿Puedo pedir información pública sobre cómo se adjudicó la licitación en la que participé y perdí?"*

**Leyes consultadas por el router:** acceso_info_publica_2008

**Respuesta:**

Esto es lo que encontré en las leyes indexadas relacionado con tu pregunta (modo LITE: extracto textual sin redacción por LLM; verifica siempre citando la fuente):



• (Ley 20.285 - Acceso a la Información Pública, Artículo 7°, Literal j)) j) Los mecanismos de participación ciudadana, en su caso.

• (Ley 20.285 - Acceso a la Información Pública, Artículo 6°) Los actos y documentos que han sido objeto de publicación en el Diario Oficial y aquellos que digan relación con las funciones, competencias y responsabilidades de los órganos de la Administración del Estado, deberán encontrarse a disposición permanente del público y en los sitios electrónicos del servicio respectivo, el que deberá llevar un registro actualizado en las oficinas de información y atención del público usuario de la Administración del Estado. De la Transparencia Activa

• (Ley 20.285 - Acceso a la Información Pública, Artículo 7°, Literal a)) a) Su estructura orgánica.

• (Ley 20.285 - Acceso a la Información Pública, Artículo 23) Los órganos de la Administración del Estado deberán mantener un índice actualizado de los actos y documentos calificados como secretos o reservados de conformidad a esta ley, en las oficinas de información o atención del público usuario de la Administración del Estado, establecidas en el decreto supremo N° 680, de 1990, del Ministerio del Interior. El índice incluirá la denominación de los actos, documentos e informaciones que sean calificados como secretos o reservados de conformidad a esta ley, y la individualización del acto o resolución en que conste tal calificación.

**Fuentes citadas:**
- Ley 20.285 - Acceso a la Información Pública, Artículo 7°, Literal j)
- Ley 20.285 - Acceso a la Información Pública, Artículo 6°
- Ley 20.285 - Acceso a la Información Pública, Artículo 7°, Literal a)
- Ley 20.285 - Acceso a la Información Pública, Artículo 23

In [12]:
result = svc.ask('Hice una promoción de "2x1 en todos los productos" por redes sociales, pero un cliente reclama que en la tienda no se la quisieron aplicar. ¿Qué dice la ley sobre publicidad y promociones?', top_k_final=4)
show_answer(result)

### 💬 Pregunta: *"Hice una promoción de "2x1 en todos los productos" por redes sociales, pero un cliente reclama que en la tienda no se la quisieron aplicar. ¿Qué dice la ley sobre publicidad y promociones?"*

**Leyes consultadas por el router:** consumidores_1997, acceso_info_publica_2008

**Respuesta:**

Esto es lo que encontré en las leyes indexadas relacionado con tu pregunta (modo LITE: extracto textual sin redacción por LLM; verifica siempre citando la fuente):



• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 36) Cuando se trate de promociones en que el incentivo consista en la participación en concursos o sorteos, el anunciante deberá informar al público sobre el monto o número de premios de aquéllos y el plazo en que se podrán reclamar. El anunciante estará obligado a difundir adecuadamente los resultados de los concursos o sorteos.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 35) En toda promoción u oferta se deberá informar al consumidor sobre las bases de la misma y el tiempo o plazo de su duración. No se entenderá cumplida esta obligación por el solo hecho de haberse depositado las bases en el oficio de un notario. En caso de rehusarse el proveedor al cumplimiento de lo ofrecido en la promoción u oferta, el consumidor podrá requerir del juez competente que ordene su cumplimiento forzado, pudiendo éste disponer una prestación equivalente en caso de no ser posible el cumplimiento en especie de lo ofrecido.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 55 D) Los proveedores que promocionen o distribuyan un contrato de adhesión de un producto o servicio financiero con sello SERNAC, sin tenerlo, o que no cumplan las obligaciones establecidas en el inciso final del artículo 55 C, serán sancionados con multa de hasta 2.250 unidades tributarias mensuales.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 55, Numeral 3) 3.- Que permitan al consumidor recurrir a un mediador o a un árbitro financiero que resuelva las controversias, quejas o reclamaciones, en el caso de que considere que el servicio de atención al cliente no ha respondido satisfactoriamente sus consultas o reclamos por cualquier producto o servicio financiero del proveedor que se otorgue en virtud de un contrato de adhesión de los señalados en el inciso siguiente. Los proveedores de productos y servicios financieros que deseen obtener el sello SERNAC deberán someter a la revisión del Servicio Nacional del Consumidor todos los contratos de adhesión que ofrezcan, relativos a los siguientes productos y servicios financieros:

**Fuentes citadas:**
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 36
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 35
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 55 D
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 55, Numeral 3

In [13]:
result = svc.ask('¿Qué garantía constitucional protege mi derecho a emprender y desarrollar libremente una actividad económica en Chile?', top_k_final=4)
show_answer(result)

### 💬 Pregunta: *"¿Qué garantía constitucional protege mi derecho a emprender y desarrollar libremente una actividad económica en Chile?"*

**Leyes consultadas por el router:** constitucion_2005, consumidores_1997

**Respuesta:**

Esto es lo que encontré en las leyes indexadas relacionado con tu pregunta (modo LITE: extracto textual sin redacción por LLM; verifica siempre citando la fuente):



• (Constitución Política de la República, Artículo 19, Numeral 21) 21º.- El derecho a desarrollar cualquiera actividad económica que no sea contraria a la moral, al orden público o a la seguridad nacional, respetando las normas legales que la regulen. El Estado y sus organismos podrán desarrollar actividades empresariales o participar en ellas sólo si una ley de quórum calificado los autoriza. En tal caso, esas actividades estarán sometidas a la legislación común aplicable a los particulares, sin perjuicio de las excepciones que por motivos justificados establezca la ley, la que deberá ser, asimismo, de quórum calificado;

• (Constitución Política de la República, Artículo 19, Inciso 107°–inciso 108°) Será de competencia exclusiva de los tribunales ordinarios de justicia declarar la extinción de tales concesiones. Las controversias que se produzcan respecto de la caducidad o extinción del dominio sobre la concesión serán resueltas por ellos; y en caso de caducidad, el afectado podrá requerir de la justicia la declaración de subsistencia de su derecho. El dominio del titular sobre su concesión minera está protegido por la garantía constitucional de que trata este número.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 8º, Literal i)) i) Efectuar, de conformidad a esta ley, cualquier otra actividad destinada a proteger, informar y educar a los consumidores.

• (Constitución Política de la República, Artículo 3º) El Estado de Chile es unitario. La administración del Estado será funcional y territorialmente descentralizada, o desconcentrada en su caso, de conformidad a la ley. Los órganos del Estado promoverán el fortalecimiento de la regionalización del país y el desarrollo equitativo y solidario entre las regiones, provincias y comunas del territorio nacional.

**Fuentes citadas:**
- Constitución Política de la República, Artículo 19, Numeral 21
- Constitución Política de la República, Artículo 19, Inciso 107°–inciso 108°
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 8º, Literal i)
- Constitución Política de la República, Artículo 3º

## 🔍 Transparencia: una limitación conocida del modo lite

El modo lite usa TF-IDF (léxico) en vez de un embedding semántico real. Esto significa que **sinónimos o formas verbales distintas de la misma raíz legal pueden no calzar tan bien** como lo haría BGE-M3. Aplicamos *stemming* en español para mitigarlo (ej. "eliminar" y "eliminación" comparten la raíz "elimin"), pero no es perfecto — sobre todo con sinónimos que no comparten raíz (ej. "borrar" vs. "suprimir").

Veamos un caso real: la Ley 21.719 usa el término técnico **"derecho de supresión"** (Artículo 7°) para lo que una PYME probablemente preguntaría como **"eliminar los datos de un cliente"**.


In [14]:
result = svc.ask("¿Qué debo hacer si un cliente pide eliminar sus datos personales?", top_k_final=8)
show_trace(result.trace)


### 🔎 Trace de recuperación para: *"¿Qué debo hacer si un cliente pide eliminar sus datos personales?"*

**1. Ruteo semántico** → leyes candidatas: `['proteccion_datos_2024', 'vida_privada_1999']`
> Router léxico -> Ley 21.719 - Protección de Datos Personales (0.53), Ley 19.628 - Protección de la Vida Privada (0.18)

**2. Búsqueda densa (coseno)** — top 5 de 15:
- `0.603` Ley 21.719 - Protección de Datos Personales, Artículo 7°, Inciso 1°
- `0.498` Ley 21.719 - Protección de Datos Personales, Artículo 1° bis, Literal s)
- `0.469` Ley 19.628 - Protección de la Vida Privada, Artículo 6°
- `0.459` Ley 21.719 - Protección de Datos Personales, Artículo 8°, Inciso 1°
- `0.423` Ley 19.628 - Protección de la Vida Privada, Artículo 12, Inciso 5°–inciso 6°

**3. Búsqueda dispersa (BM25)** — top 5 de 13:
- `10.118` Ley 19.628 - Protección de la Vida Privada, Artículo 12, Inciso 1°–inciso 4°
- `9.616` Ley 19.628 - Protección de la Vida Privada, Artículo 6°
- `9.254` Ley 21.719 - Protección de Datos Personales, Artículo 7°, Inciso 1°
- `8.936` Ley 21.719 - Protección de Datos Personales, Artículo 1° bis, Literal s)
- `8.477` Ley 21.719 - Protección de Datos Personales, Artículo 1° bis, Literal i)

**4. Fusión RRF** — top 5 de 12:
- `0.0323` Ley 21.719 - Protección de Datos Personales, Artículo 7°, Inciso 1°
- `0.0320` Ley 19.628 - Protección de la Vida Privada, Artículo 6°
- `0.0318` Ley 21.719 - Protección de Datos Personales, Artículo 1° bis, Literal s)
- `0.0315` Ley 19.628 - Protección de la Vida Privada, Artículo 12, Inciso 1°–inciso 4°
- `0.0299` Ley 19.628 - Protección de la Vida Privada, Artículo 12, Inciso 5°–inciso 6°

**5. Reranker** — contexto final (8 chunks):
- `1.020` **Ley 21.719 - Protección de Datos Personales, Artículo 7°, Inciso 1°** (dense#1, sparse#3, rrf=0.0323)
- `0.510` **Ley 19.628 - Protección de la Vida Privada, Artículo 6°** (dense#3, sparse#2, rrf=0.0320)
- `0.351` **Ley 21.719 - Protección de Datos Personales, Artículo 1° bis, Literal s)** (dense#2, sparse#4, rrf=0.0318)
- `0.258` **Ley 19.628 - Protección de la Vida Privada, Artículo 12, Inciso 1°–inciso 4°** (dense#6, sparse#1, rrf=0.0315)
- `0.206` **Ley 19.628 - Protección de la Vida Privada, Artículo 12, Inciso 5°–inciso 6°** (dense#5, sparse#9, rrf=0.0299)
- `0.179` **Ley 21.719 - Protección de Datos Personales, Artículo tercero (transitorio)** (dense#12, sparse#7, rrf=0.0288)
- `0.154` **Ley 21.719 - Protección de Datos Personales, Artículo 8°, Inciso 1°** (dense#4, sparse#-, rrf=0.0156)
- `0.132` **Ley 21.719 - Protección de Datos Personales, Artículo 1° bis, Literal i)** (dense#-, sparse#5, rrf=0.0154)

Como se ve arriba, el **Artículo 7° (Derecho de supresión)** sí aparece en los resultados (el ruteo semántico y BM25/TF-IDF con stemming lo encuentran), pero no siempre en el primer lugar — compite con otros artículos relacionados a "datos personales" en general. En producción, el embedding semántico de **BGE-M3** entendería que "eliminar datos" y "derecho de supresión" son la misma idea con mucha más precisión, y el **reranker BGE** (cross-encoder, mira pregunta+chunk juntos) subiría el artículo correcto con más confianza que nuestra heurística léxica.

**Esto es exactamente el tipo de caso que vale la pena revisar juntos**: si al probar el notebook encuentras preguntas donde la respuesta correcta existe en el corpus pero no aparece en el top-k, es señal de que necesitamos (a) ajustar los pesos del router/reranker lite, o (b) confirma que vale la pena priorizar activar el modo producción con BGE-M3 real.


## 🧪 Prueba tú mismo (modo interactivo)

Cambia la pregunta en la celda de abajo y vuelve a correrla para simular cualquier otra consulta de cliente PYME.


In [15]:
mi_pregunta = "¿Qué pasa si un proveedor no cumple con la garantía legal de un producto?"

result = svc.ask(mi_pregunta, top_k_final=4)
show_answer(result)


### 💬 Pregunta: *"¿Qué pasa si un proveedor no cumple con la garantía legal de un producto?"*

**Leyes consultadas por el router:** consumidores_1997

**Respuesta:**

Esto es lo que encontré en las leyes indexadas relacionado con tu pregunta (modo LITE: extracto textual sin redacción por LLM; verifica siempre citando la fuente):



• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 17 M) Los proveedores de productos o servicios financieros pactados por contrato de adhesión garantizados por cualquier tipo de garantía estarán obligados a conservar, a lo menos de manera digital, y durante el tiempo de existencia de la garantía en su favor, todos los documentos en que consten dichas garantías.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 50 D) En aquellos casos en los que en virtud de esta ley se interponga demanda en contra de una persona jurídica, su notificación se efectuará al representante legal de ésta o bien al jefe del local donde se compró el producto o se prestó el servicio. Será obligación de todos los proveedores exhibir en un lugar visible del local la individualización completa de quien cumpla la función de jefe del local, indicándose al menos el nombre completo y su domicilio.

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 17 H, Inciso 1°) Los proveedores de productos o servicios financieros no podrán ofrecer o vender productos o servicios de manera atada. Se entiende que un producto o servicio financiero es vendido en forma atada si el proveedor:

• (Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 14) Cuando con conocimiento del proveedor se expendan productos con alguna deficiencia, usados o refaccionados o cuando se ofrezcan productos en cuya fabricación o elaboración se hayan utilizado partes o piezas usadas, se deberán informar de manera expresa las circunstancias antes mencionadas al consumidor, antes de que éste decida la operación de compra. Será bastante constancia el usar en los propios artículos, en sus envoltorios, en avisos o carteles visibles en sus locales de atención al público las expresiones "segunda selección", "hecho con materiales usados" u otras equivalentes. El cumplimiento de lo dispuesto en el inciso anterior eximirá al proveedor de las obligaciones derivadas del derecho de opción que se establece en los artículos 19 y 20, sin perjuicio de aquellas que hubiera contraído el proveedor en virtud de la garantía otorgada al producto.

**Fuentes citadas:**
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 17 M
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 50 D
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 17 H, Inciso 1°
- Ley 19.496 - Protección de los Derechos de los Consumidores, Artículo 14

## Cómo pasar a modo producción (BGE-M3 + Qwen3 + BGE FlagReranker)

En una máquina con GPU (o CPU con más RAM) y acceso a Internet para descargar modelos de HuggingFace:

```bash
pip install -r requirements-production.txt   # torch, transformers, FlagEmbedding, accelerate

export RAG_MODE=production
python scripts/01_ingest_and_index.py        # re-indexa con embeddings BGE-M3 reales
jupyter notebook notebooks/01_pipeline_demo.ipynb
```

No hay que tocar ni una línea del pipeline: `config.RAG_MODE` (o la variable de entorno `RAG_MODE`) es el único interruptor. Las clases `BGEM3Embedder`, `BGERerankerProduction`, `QwenRouterProduction` y `QwenGeneratorProduction` en `leyes_rag/` ya están implementadas siguiendo exactamente la arquitectura acordada (BGE-M3 + BM25 + fusión RRF + BGE FlagReranker + Qwen3 para ruteo y generación).

Si en vez de cargar los modelos en el propio proceso Python prefieres un servidor **Ollama** o **vLLM** para Qwen3, basta con setear `LLM_BACKEND=ollama` (o `vllm`) y `OLLAMA_HOST` — ver `leyes_rag/config.py::ModelConfig`.

### Próximos pasos sugeridos
1. Revisar conmigo las respuestas de este notebook: ¿alguna cita está mal, incompleta o el chunking cortó un artículo de forma rara?
2. Decidir si conviene ampliar el catálogo de leyes (`leyes_rag/config.py::LAW_CATALOG`).
3. Cuando tengamos acceso a GPU/Internet, activar `RAG_MODE=production` y comparar la calidad real de BGE-M3/Qwen3 contra este baseline lite.
